In [1]:
import pandas as pd
from datetime import datetime

# Loading Data

In [2]:
# Load features dataset
features_df = pd.read_csv(r'C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\all_league_bowl_features.csv')
features_df['date'] = pd.to_datetime(features_df['date'])

# Load original batting data
bowling_df = pd.read_csv(r'C:\Users\HP\jupyter notebook files\FIFS_IPL\data\mbm\all_league\all_league_bowl_final.csv')
bowling_df['date'] = pd.to_datetime(bowling_df['date'])

# Filling NULL values for league debuts

In [8]:
def fill_all_bowling_nulls(features_df, bowling_df):
    """
    Fill NULL values in bowling features dataframe for:
    1. League debuts (where career stats are NULL)
    2. Season debuts (where season stats are NULL)
    3. Venue debuts (where venue stats are NULL)
    4. Innings debuts (where innings stats are NULL)
    """
    # Create a copy to avoid modifying the original
    features_df = features_df.copy()

    # Ensure proper date format
    bowling_df['date'] = pd.to_datetime(bowling_df['date'])
    features_df['date'] = pd.to_datetime(features_df['date'])

    # Pre-process bowling data once
    print("Pre-processing bowling data...")
    bowling_df = bowling_df.copy()

    # Add season column
    bowling_df['season'] = bowling_df['date'].dt.year

    # Convert overs to balls
    # Replace 'DNB' with '0.0' and ensure overs is a string
    bowling_df['overs'] = bowling_df['overs'].replace('DNB', '0.0').astype(str)
    # Split overs into two columns
    split_cols = bowling_df['overs'].str.split('.', expand=True)
    # Fill missing values with '0' to avoid None or NaN values
    split_cols = split_cols.fillna('0')
    # Convert the split columns to integer and assign to new columns
    bowling_df[['overs_int', 'balls_int']] = split_cols.astype(int)
    # Continue with further calculations
    bowling_df['total_balls'] = bowling_df['overs_int'] * 6 + bowling_df['balls_int']
    bowling_df['total_overs'] = bowling_df['total_balls'] / 6

    # Add dot balls column if it doesn't exist
    if 'dots' not in bowling_df.columns:
        bowling_df['dots'] = 0

    # Define minimum match threshold for statistical significance
    MIN_MATCHES = 3

    # Get unique player IDs
    unique_players = features_df['bowler_id'].unique()
    print(f"Processing {len(unique_players)} unique players")

    # Counters for tracking updates
    updates_made = 0
    null_cases = {'league_debuts': 0, 'season_debuts': 0, 'venue_debuts': 0, 'innings_debuts': 0}

    # Process each player
    for i, player_id in enumerate(unique_players):
        if (i+1) % 100 == 0 or i+1 == len(unique_players):
            print(f"Processed {i+1}/{len(unique_players)} players")

        # Get player data
        player_features = features_df[features_df['bowler_id'] == player_id]

        # Skip if player only has one entry (absolute debut)
        if len(player_features) <= 1:
            continue

        # Get player's first match date
        first_match_date = player_features['date'].min()

        # Find rows with NULL values in any of the categories
        null_careers = player_features[
            (pd.isna(player_features['career_avg'])) &
            (player_features['date'] > first_match_date)
        ]

        null_seasons = player_features[
            pd.isna(player_features['ssn_avg']) &
            ~pd.isna(player_features['career_avg'])  # Only include rows where career is NOT null
        ]

        null_venues = player_features[
            pd.isna(player_features['venue_avg']) &
            ~pd.isna(player_features['career_avg'])
        ]

        null_innings = player_features[
            pd.isna(player_features['inns_avg']) &
            ~pd.isna(player_features['career_avg'])
        ]

        # Update counters
        null_cases['league_debuts'] += len(null_careers)
        null_cases['season_debuts'] += len(null_seasons)
        null_cases['venue_debuts'] += len(null_venues)
        null_cases['innings_debuts'] += len(null_innings)

        # Combine all null rows for efficient processing
        all_null_rows = pd.concat([null_careers, null_seasons, null_venues, null_innings]).drop_duplicates()

        if len(all_null_rows) == 0:
            continue

        # Get bowling data for this player and sort by date
        player_bowling = bowling_df[bowling_df['bowler_id'] == player_id].sort_values('date')

        # Process each null row
        for idx, null_row in all_null_rows.iterrows():
            current_date = null_row['date']
            current_league = null_row['league']
            current_season = null_row['season']
            current_venue = null_row['venue']
            current_inns = null_row['inns']

            # Get all matches before this date
            prior_matches = player_bowling[player_bowling['date'] < current_date]

            if len(prior_matches) == 0:
                continue

            updates_made += 1

            # Calculate aggregated stats once
            total_wickets = prior_matches['wickets'].sum()
            total_runs = prior_matches['runs'].sum()
            total_matches = len(prior_matches)
            total_balls = prior_matches['total_balls'].sum()
            total_overs = prior_matches['total_overs'].sum()
            total_dots = prior_matches['dots'].sum()

            # Career stats
            career_avg = total_runs / max(total_wickets, 1)
            career_eco = total_runs / max(total_overs, 1)
            career_sr = total_balls / max(total_wickets, 1)
            career_wpm = total_wickets / max(total_matches, 1)
            career_opm = total_overs / max(total_matches, 1)
            career_dotpct = total_dots / max(total_balls, 1)

            # FILL CAREER STATS IF NEEDED
            if pd.isna(features_df.loc[idx, 'career_avg']):
                features_df.loc[idx, 'career_avg'] = career_avg
                features_df.loc[idx, 'career_eco'] = career_eco
                features_df.loc[idx, 'career_sr'] = career_sr
                features_df.loc[idx, 'career_wpm'] = career_wpm
                features_df.loc[idx, 'career_opm'] = career_opm

                if 'career_dotpct' in features_df.columns:
                    features_df.loc[idx, 'career_dotpct'] = career_dotpct

            # FILL INNINGS STATS IF NEEDED
            if pd.isna(features_df.loc[idx, 'inns_avg']):
                # Try to get innings-specific prior matches
                prior_inns_matches = prior_matches[prior_matches['inns'] == current_inns]

                if len(prior_inns_matches) > 0:
                    # Use innings-specific stats
                    inns_wickets = prior_inns_matches['wickets'].sum()
                    inns_runs = prior_inns_matches['runs'].sum()
                    inns_matches = len(prior_inns_matches)
                    inns_balls = prior_inns_matches['total_balls'].sum()
                    inns_overs = prior_inns_matches['total_overs'].sum()
                    inns_dots = prior_inns_matches['dots'].sum()

                    inns_avg = inns_runs / max(inns_wickets, 1)
                    inns_eco = inns_runs / max(inns_overs, 1)
                    inns_sr = inns_balls / max(inns_wickets, 1)
                    inns_wpm = inns_wickets / max(inns_matches, 1)
                    inns_opm = inns_overs / max(inns_matches, 1)
                    inns_dotpct = inns_dots / max(inns_balls, 1)
                else:
                    # Fallback to career stats
                    inns_avg = career_avg
                    inns_eco = career_eco
                    inns_sr = career_sr
                    inns_wpm = career_wpm
                    inns_opm = career_opm
                    inns_dotpct = career_dotpct

                features_df.loc[idx, 'inns_avg'] = inns_avg
                features_df.loc[idx, 'inns_eco'] = inns_eco
                features_df.loc[idx, 'inns_sr'] = inns_sr
                features_df.loc[idx, 'inns_wpm'] = inns_wpm
                features_df.loc[idx, 'inns_opm'] = inns_opm

                if 'inns_dotpct' in features_df.columns:
                    features_df.loc[idx, 'inns_dotpct'] = inns_dotpct

            # FILL VENUE STATS IF NEEDED
            if pd.isna(features_df.loc[idx, 'venue_avg']):
                # Try to get venue-specific prior matches
                prior_venue_matches = prior_matches[prior_matches['venue'] == current_venue]

                if len(prior_venue_matches) > 0:
                    # Use venue-specific stats
                    venue_wickets = prior_venue_matches['wickets'].sum()
                    venue_runs = prior_venue_matches['runs'].sum()
                    venue_matches = len(prior_venue_matches)
                    venue_balls = prior_venue_matches['total_balls'].sum()
                    venue_overs = prior_venue_matches['total_overs'].sum()
                    venue_dots = prior_venue_matches['dots'].sum()

                    venue_avg = venue_runs / max(venue_wickets, 1)
                    venue_eco = venue_runs / max(venue_overs, 1)
                    venue_sr = venue_balls / max(venue_wickets, 1)
                    venue_wpm = venue_wickets / max(venue_matches, 1)
                    venue_opm = venue_overs / max(venue_matches, 1)
                    venue_dotpct = venue_dots / max(venue_balls, 1)
                else:
                    # Fallback to career stats
                    venue_avg = career_avg
                    venue_eco = career_eco
                    venue_sr = career_sr
                    venue_wpm = career_wpm
                    venue_opm = career_opm
                    venue_dotpct = career_dotpct

                features_df.loc[idx, 'venue_avg'] = venue_avg
                features_df.loc[idx, 'venue_eco'] = venue_eco
                features_df.loc[idx, 'venue_sr'] = venue_sr
                features_df.loc[idx, 'venue_wpm'] = venue_wpm
                features_df.loc[idx, 'venue_opm'] = venue_opm

                if 'venue_dotpct' in features_df.columns:
                    features_df.loc[idx, 'venue_dotpct'] = venue_dotpct

            # FILL SEASON STATS IF NEEDED - UPDATED LOGIC
            if pd.isna(features_df.loc[idx, 'ssn_avg']):
                # Check if player has played in this league before
                prior_same_league = prior_matches[prior_matches['league'] == current_league]

                if len(prior_same_league) > 0:
                    # 1. For players who have played in this league before:
                    # Use only the most recent previous season in the same league
                    seasons = sorted(prior_same_league['season'].unique())
                    prev_seasons = [s for s in seasons if s < current_season]

                    if prev_seasons:
                        # Get most recent previous season
                        prev_season = max(prev_seasons)
                        prev_season_matches = prior_same_league[prior_same_league['season'] == prev_season]

                        # Calculate season stats based on prev_season_matches
                        season_wickets = prev_season_matches['wickets'].sum()
                        season_runs = prev_season_matches['runs'].sum()
                        season_matches = len(prev_season_matches)
                        season_balls = prev_season_matches['total_balls'].sum()
                        season_overs = prev_season_matches['total_overs'].sum()
                        season_dots = prev_season_matches['dots'].sum()
                    else:
                        # Fallback to career stats (shouldn't happen often)
                        season_wickets = total_wickets
                        season_runs = total_runs
                        season_matches = total_matches
                        season_balls = total_balls
                        season_overs = total_overs
                        season_dots = total_dots

                else:
                    # 2. For players in their first season in this league:
                    # First get all leagues this player has played in
                    leagues_played = prior_matches['league'].unique()
                    data_found = False

                    # First Priority: Most Recently Played League with Sufficient Matches
                    for league in sorted(leagues_played, key=lambda l: prior_matches[prior_matches['league'] == l]['date'].max(), reverse=True):
                        league_matches = prior_matches[prior_matches['league'] == league]

                        # Check if player has sufficient matches in this league
                        if len(league_matches) >= MIN_MATCHES:
                            # Use this league's data
                            season_wickets = league_matches['wickets'].sum()
                            season_runs = league_matches['runs'].sum()
                            season_matches = len(league_matches)
                            season_balls = league_matches['total_balls'].sum()
                            season_overs = league_matches['total_overs'].sum()
                            season_dots = league_matches['dots'].sum()
                            data_found = True
                            break

                    # Second Priority: Current Season Data from Other Leagues (if no sufficient league found)
                    if not data_found:
                        current_season_other_leagues = prior_matches[prior_matches['season'] == current_season]

                        if len(current_season_other_leagues) > 0:
                            # Use all current season data from any league
                            season_wickets = current_season_other_leagues['wickets'].sum()
                            season_runs = current_season_other_leagues['runs'].sum()
                            season_matches = len(current_season_other_leagues)
                            season_balls = current_season_other_leagues['total_balls'].sum()
                            season_overs = current_season_other_leagues['total_overs'].sum()
                            season_dots = current_season_other_leagues['dots'].sum()
                        else:
                            # Last resort: Fall back to career stats
                            season_wickets = total_wickets
                            season_runs = total_runs
                            season_matches = total_matches
                            season_balls = total_balls
                            season_overs = total_overs
                            season_dots = total_dots

                # Calculate and update season stats
                ssn_avg = season_runs / max(season_wickets, 1)
                ssn_eco = season_runs / max(season_overs, 1)
                ssn_sr = season_balls / max(season_wickets, 1)
                ssn_wpm = season_wickets / max(season_matches, 1)
                ssn_opm = season_overs / max(season_matches, 1)
                ssn_dotpct = season_dots / max(season_balls, 1)

                features_df.loc[idx, 'ssn_avg'] = ssn_avg
                features_df.loc[idx, 'ssn_eco'] = ssn_eco
                features_df.loc[idx, 'ssn_sr'] = ssn_sr
                features_df.loc[idx, 'ssn_wpm'] = ssn_wpm
                features_df.loc[idx, 'ssn_opm'] = ssn_opm

                if 'ssn_dotpct' in features_df.columns:
                    features_df.loc[idx, 'ssn_dotpct'] = ssn_dotpct

    # Print summary
    print(f"\nSummary of NULL value fills:")
    for case_type, count in null_cases.items():
        print(f"- {case_type}: {count}")
    print(f"Total updates made: {updates_made}")

    return features_df

In [21]:
updated_features_df = fill_all_bowling_nulls(features_df, bowling_df)

Pre-processing bowling data...
Processing 4691 unique players
Processed 100/4691 players
Processed 200/4691 players
Processed 300/4691 players
Processed 400/4691 players
Processed 500/4691 players
Processed 600/4691 players
Processed 700/4691 players
Processed 800/4691 players
Processed 900/4691 players
Processed 1000/4691 players
Processed 1100/4691 players
Processed 1200/4691 players
Processed 1300/4691 players
Processed 1400/4691 players
Processed 1500/4691 players
Processed 1600/4691 players
Processed 1700/4691 players
Processed 1800/4691 players
Processed 1900/4691 players
Processed 2000/4691 players
Processed 2100/4691 players
Processed 2200/4691 players
Processed 2300/4691 players
Processed 2400/4691 players
Processed 2500/4691 players
Processed 2600/4691 players
Processed 2700/4691 players
Processed 2800/4691 players
Processed 2900/4691 players
Processed 3000/4691 players
Processed 3100/4691 players
Processed 3200/4691 players
Processed 3300/4691 players
Processed 3400/4691 pla

# Final processing

In [22]:
updated_features_df['date'] = pd.to_datetime(features_df['date'])
updated_features_df.sort_values(['date', 'inns'], inplace=True)

In [23]:
updated_features_df.career_avg.isnull().sum()

4691

In [24]:
def add_career_debut_column(df):
    """
    Adds a career_debut column to the dataframe.
    Sets value to 1 where career_avg is null (indicating league debut),
    and 0 for all other rows.

    Args:
        df: DataFrame containing batting or bowling features

    Returns:
        DataFrame with the additional career_debut column
    """
    # Create a copy to avoid modifying the original
    df = df.copy()

    # Create career_debut column based on whether career_avg is null
    df['career_debut'] = pd.isna(df['career_avg']).astype(int)

    print(f"Added career_debut column. Found {df['career_debut'].sum()} career debuts.")

    return df


# After creating your features dataframe
updated_features_df = add_career_debut_column(updated_features_df)

Added career_debut column. Found 4691 career debuts.


In [25]:
def convert_nan_to_zero(df):
    """
    Convert all NaN values in the dataframe to 0.

    Args:
        df: DataFrame containing features with potential NaN values

    Returns:
        DataFrame with NaN values replaced by 0
    """
    # Count NaN values before conversion
    nan_count_before = df.isna().sum().sum()

    # Make a copy to avoid modifying the original dataframe
    df_copy = df.copy()

    # Replace all NaN values with 0
    df_copy = df_copy.fillna(0)

    # Verify all NaNs have been replaced
    nan_count_after = df_copy.isna().sum().sum()

    print(f"Replaced {nan_count_before} NaN values with 0")
    print(f"Remaining NaN values: {nan_count_after}")

    return df_copy

# Usage:
updated_features_df = convert_nan_to_zero(updated_features_df)

Replaced 117375 NaN values with 0
Remaining NaN values: 0


In [14]:
# Define the cutoff date
cutoff_date = pd.to_datetime('2025-03-21')

# Filter the dataset to include only rows with dates on or before 21 march 2025
updated_features_df = updated_features_df[updated_features_df['date'] <= cutoff_date]

In [26]:
min(updated_features_df.date), max(updated_features_df.date)

(Timestamp('2005-02-17 00:00:00'), Timestamp('2025-04-07 00:00:00'))

In [27]:
updated_features_df.columns

Index(['date', 'match_id', 'bowl_team', 'bat_team', 'bowler_id', 'bowler',
       'inns', 'venue', 'season', 'league', 'career_avg', 'career_eco',
       'career_sr', 'career_wpm', 'career_opm', 'inns_avg', 'inns_eco',
       'inns_sr', 'inns_wpm', 'inns_opm', 'venue_avg', 'venue_eco', 'venue_sr',
       'venue_wpm', 'venue_opm', 'ssn_avg', 'ssn_eco', 'ssn_sr', 'ssn_wpm',
       'ssn_opm', 'last5_avg', 'last5_eco', 'last5_sr', 'last5_wpm',
       'last5_opm', 'career_debut'],
      dtype='object')

In [28]:
updated_features_df = updated_features_df.reindex(columns=['date', 'match_id', 'bowler', 'bowler_id', 'bowl_team', 'bat_team',
       'league', 'career_avg', 'career_eco', 'career_sr', 'career_wpm', 'career_opm',
       'inns', 'inns_avg', 'inns_eco', 'inns_sr', 'inns_wpm', 'inns_opm',
       'venue', 'venue_avg', 'venue_eco', 'venue_sr', 'venue_wpm', 'venue_opm',
       'season', 'ssn_avg', 'ssn_eco', 'ssn_sr', 'ssn_wpm', 'ssn_opm',
       'last5_avg', 'last5_eco', 'last5_sr', 'last5_wpm', 'last5_opm',
       'career_debut'])

In [29]:
updated_features_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 161041 entries, 9431 to 159380
Data columns (total 36 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   date          161041 non-null  datetime64[ns]
 1   match_id      161041 non-null  int64         
 2   bowler        161041 non-null  object        
 3   bowler_id     161041 non-null  int64         
 4   bowl_team     161041 non-null  object        
 5   bat_team      161041 non-null  object        
 6   league        161041 non-null  object        
 7   career_avg    161041 non-null  float64       
 8   career_eco    161041 non-null  float64       
 9   career_sr     161041 non-null  float64       
 10  career_wpm    161041 non-null  float64       
 11  career_opm    161041 non-null  float64       
 12  inns          161041 non-null  int64         
 13  inns_avg      161041 non-null  float64       
 14  inns_eco      161041 non-null  float64       
 15  inns_sr       

In [30]:
updated_features_df.to_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\null_val_handled_data\all_league_bowl_features_dr.csv", index=False)